# LoCI LAMP Product Creation - Setup Phase

This notebook sets up all the necessary components for creating the LoCI LAMP product with DPP.

**Flow mirrors the GUI CreateProjectForm.tsx:**
1. User authentication and registration
2. Location setup
3. Units and resource specifications
4. Process definitions
5. Initial resources and components creation
6. Image asset preparation

In [1]:
# Module imports and auto-reload setup
%load_ext autoreload
%aimport if_lib, if_utils, if_dpp, if_graphics, if_consts, if_gc1dpp
%autoreload 1
import os
import json
import random
import csv
from pathlib import Path

from if_utils import get_filename, show_data, save_traces

from if_lib import generate_random_challenge, read_HMAC, read_keypair, get_id_person, get_location_id, \
get_unit_id, get_resource_spec_id, get_resource, get_process, create_event, make_transfer, reduce_resource, set_user_location

from if_gc1dpp import upload_file_on_dpp, calculate_file_checksum

## Configuration and Endpoints

In [2]:
# Define constants for this use case
USE_CASE = 'locilamp'

# Zenflows API endpoint
ENDPOINT = 'https://proxy.dpp-staging.dyne.im/zenflows/api'

# DPP service endpoint
DPP_URL = 'https://proxy.dpp-staging.dyne.im/interfacer-dpp'

# CSV file with product data
CSV_FILE = Path('/Users/alcibiade/dyne/if/20260127 LOCI LAMP product informaiton DPP DE.csv')

# Assets directory
ASSETS_DIR = Path('/Users/alcibiade/dyne/if/Interfacer-notebook/assets')

# Participants for LoCI LAMP production
USERS = ['tchibo', 'locilamp_designer', 'locilamp_manufacturer']

print(f"Configuration:")
print(f"  Use Case: {USE_CASE}")
print(f"  Zenflows: {ENDPOINT}")
print(f"  DPP URL: {DPP_URL}")
print(f"  CSV File: {CSV_FILE}")
print(f"  Assets: {ASSETS_DIR}")

Configuration:
  Use Case: locilamp
  Zenflows: https://proxy.dpp-staging.dyne.im/zenflows/api
  DPP URL: https://proxy.dpp-staging.dyne.im/interfacer-dpp
  CSV File: /Users/alcibiade/dyne/if/20260127 LOCI LAMP product informaiton DPP DE.csv
  Assets: /Users/alcibiade/dyne/if/Interfacer-notebook/assets


## File Path Configuration

In [3]:
# Calculate names of settings files
USERS_FILE = get_filename('cred_users.json', ENDPOINT, USE_CASE)
LOCS_FILE = get_filename('loc_users.json', ENDPOINT, USE_CASE)
UNITS_FILE = get_filename('units_data.json', ENDPOINT, USE_CASE)
SPECS_FILE = get_filename('res_spec_data.json', ENDPOINT, USE_CASE)
DPP_FILE = get_filename('dpp_data.json', ENDPOINT, USE_CASE)
RES_FILE = get_filename('initial_resources.json', ENDPOINT, USE_CASE)
PROCESS_FILE = get_filename('process_data.json', ENDPOINT, USE_CASE)
IMAGES_FILE = get_filename('images_data.json', ENDPOINT, USE_CASE)

print(f"Data will be saved to:")
print(f"  Users: {USERS_FILE}")
print(f"  Locations: {LOCS_FILE}")
print(f"  Units: {UNITS_FILE}")
print(f"  Resource Specs: {SPECS_FILE}")
print(f"  DPP Data: {DPP_FILE}")
print(f"  Initial Resources: {RES_FILE}")
print(f"  Processes: {PROCESS_FILE}")
print(f"  Images: {IMAGES_FILE}")

Data will be saved to:
  Users: use_cases/locilamp/proxy.dpp-staging.dyne.im%2Fzenflows%2Fapi/cred_users.json
  Locations: use_cases/locilamp/proxy.dpp-staging.dyne.im%2Fzenflows%2Fapi/loc_users.json
  Units: use_cases/locilamp/proxy.dpp-staging.dyne.im%2Fzenflows%2Fapi/units_data.json
  Resource Specs: use_cases/locilamp/proxy.dpp-staging.dyne.im%2Fzenflows%2Fapi/res_spec_data.json
  DPP Data: use_cases/locilamp/proxy.dpp-staging.dyne.im%2Fzenflows%2Fapi/dpp_data.json
  Initial Resources: use_cases/locilamp/proxy.dpp-staging.dyne.im%2Fzenflows%2Fapi/initial_resources.json
  Processes: use_cases/locilamp/proxy.dpp-staging.dyne.im%2Fzenflows%2Fapi/process_data.json
  Images: use_cases/locilamp/proxy.dpp-staging.dyne.im%2Fzenflows%2Fapi/images_data.json


## Load Product Data from CSV

Parse the LOCI LAMP product information from the CSV file.

In [4]:
# Parse LOCI LAMP CSV file
def parse_loci_lamp_csv(csv_path):
    """Parse LOCI LAMP CSV file into structured product data."""
    product_data = {}
    current_category = None
    
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter=';')
        next(reader)  # Skip header
        for row in reader:
            if len(row) < 4:
                continue
            category = row[0].strip() if row[0].strip() else current_category
            if row[0].strip():
                current_category = category
            title = row[1].strip() if len(row) > 1 else ''
            value = row[3].strip() if len(row) > 3 else ''
            
            if title and value:
                if category not in product_data:
                    product_data[category] = {}
                product_data[category][title] = value
    
    return product_data

# Load the CSV data
print("Loading LOCI LAMP product data from CSV...")
loci_lamp_data = parse_loci_lamp_csv(CSV_FILE)

print(f"\n✓ Loaded {len(loci_lamp_data)} categories:")
for category in loci_lamp_data:
    print(f"  - {category}: {len(loci_lamp_data[category])} fields")

# Extract key product info
product_overview = loci_lamp_data.get('Product Overview', {})
print(f"\nProduct Information:")
print(f"  Brand: {product_overview.get('Brand Name', 'N/A')}")
print(f"  Name: {product_overview.get('Product Name', 'N/A')}")
print(f"  Model: {product_overview.get('Model Name', 'N/A')}")

Loading LOCI LAMP product data from CSV...

✓ Loaded 9 categories:
  - Product Overview: 12 fields
  - Repairability: 2 fields
  - Environmental Impact: 2 fields
  - Compliance and Standards: 2 fields
  - Certificates: 1 fields
  - Recyclability: 2 fields
  - Energy Use & Efficiency: 5 fields
  - Component Information Drill: 1 fields
  - Economic Operator: 4 fields

Product Information:
  Brand: Tchibo GmbH
  Name: LOCI LAMP
  Model: LOCILAMP V 2.0


## Initialize Data Structures

In [5]:
# Create data structures
process_data = {}
res_data = {}
event_seq = []
dpp_data = {}
images_data = {}

# Initialize or load user data - LoCI LAMP specific users
if os.path.isfile(USERS_FILE):
    with open(USERS_FILE,'r') as f:
        users_data = json.loads(f.read())
    print("Credentials file available for users")
else:
    users_data = {}
    # Tchibo - the brand owner
    users_data['tchibo'] = {
      "userChallenges": {
        "whereParentsMet": "Hamburg",
        "nameFirstPet": "Kaffee",
        "nameFirstTeacher": "Hans",
        "whereHomeTown": "Hamburg",
        "nameMotherMaid": "Schmidt"
      },
      "name": "Tchibo GmbH",
      "username": "tchibo_locilamp",
      "email": "service@tchibo.de",
      "note": "Tchibo GmbH - LOCI LAMP brand"
    }
    # Designer of the lamp
    users_data['locilamp_designer'] = {
      "userChallenges": {
        "whereParentsMet": "Berlin",
        "nameFirstPet": "Licht",
        "nameFirstTeacher": "Maria",
        "whereHomeTown": "Munich",
        "nameMotherMaid": "Weber"
      },
      "name": "LoCI LAMP Designer",
      "username": "locilamp_designer",
      "email": "designer@locilamp.de",
      "note": "LOCI LAMP product designer"
    }
    # Manufacturer
    users_data['locilamp_manufacturer'] = {
      "userChallenges": {
        "whereParentsMet": "Frankfurt",
        "nameFirstPet": "Holz",
        "nameFirstTeacher": "Peter",
        "whereHomeTown": "Stuttgart",
        "nameMotherMaid": "Müller"
      },
      "name": "LoCI LAMP Manufacturer",
      "username": "locilamp_manufacturer",
      "email": "manufacture@locilamp.de",
      "note": "LOCI LAMP manufacturer"
    }
    with open(USERS_FILE,'w') as f:
        json.dump(users_data, f)
    print("Created new user credentials file")

# Initialize or load location data
if os.path.isfile(LOCS_FILE):
    with open(LOCS_FILE,'r') as f:
        locs_data = json.loads(f.read())
    print("Location file available")
else:
    locs_data = {}
    # Tchibo HQ in Hamburg
    locs_data['tchibo'] = {
        "name": "Tchibo GmbH Headquarters",
        "lat": 53.6040379,
        "long": 10.0221277,
        "addr": "Überseering 18, 22297 Hamburg, Germany",
        "note": "Tchibo headquarters"
    }
    # Designer location
    locs_data['locilamp_designer'] = {
        "name": "LoCI LAMP Design Studio",
        "lat": 52.5200066,
        "long": 13.404954,
        "addr": "Design Studio, Berlin, Germany",
        "note": "LOCI LAMP design location"
    }
    # Manufacturer location in Germany
    locs_data['locilamp_manufacturer'] = {
        "name": "LoCI LAMP Manufacturing",
        "lat": 48.7758459,
        "long": 9.1829321,
        "addr": "Manufacturing Facility, Stuttgart, Germany",
        "note": "LOCI LAMP manufacturing location"
    }
    with open(LOCS_FILE,'w') as f:
        json.dump(locs_data, f)
    print("Created new location file")

# Initialize units and specs
if os.path.isfile(UNITS_FILE):
    with open(UNITS_FILE,'r') as f:
        units_data = json.loads(f.read())
    print(f"Unit file available")
else:
    units_data = {}

if os.path.isfile(SPECS_FILE):
    with open(SPECS_FILE,'r') as f:
        res_spec_data = json.loads(f.read())
    print(f"Resource Spec file available")
else:
    res_spec_data = {}

Created new user credentials file
Created new location file


## Authentication Setup: HMAC Generation

In [6]:
# Read HMAC or get it from the server
for user in USERS:
    read_HMAC(USERS_FILE, users_data, user, endpoint=ENDPOINT)

## Cryptographic Key Generation

In [7]:
# Read the keypair for each user
for user in USERS:
    read_keypair(USERS_FILE, users_data, user)

result: ZenResult(output='{"ecdh_public_key":"BG58/cOl/NfuJ6Kz5PAxUqHAu0UTwNQtOZjg1kBbY/BHAdrZqOo9gDrfxH+XNVYyfKQr+DfhsADaQs2/4ge5DKY=","eddsa_public_key":"3hpGAbEpBNas1pjua21c64uqFhTB8EbVJThtipjBwYAH","ethereum_address":"0x113F17C6cb6f0b0C710C89a29587848e1e588252","keyring":{"ecdh":"bWE3VJ1Jjh8IjBzSPuFO8aUXBc+2Sx0rwMA+VBGO2kI=","eddsa":"DMkvjVSPoGkbMWeVQK3Xgahiqgv8GydQG6Fqs69axjK1","ethereum":"8ae12e6ecab858c6a97e26e8267582d4eaf8b7c23a1915a87714bbc926dbdb40","reflow":"iKXetJqoss/J0TvEjAJP0KjcbPX+SJ9JuQgE5WmaZeA=","schnorr":"JuJOc58yzLuXeMb/Mm1h0rixGS+MKv7eRBnNi2Y8/Vg="},"reflow_public_key":"AN2sm2L22zsEhQNk+wm9uFF/O2m3pZwUcRFR3H7V04+mACRMl/GFPXNHvI5VOeCzEfgtdj8V8qara3i6gP2XY4h8I+/OmIiU+uwXXR6y+du50wOivSw/DEN9GsREstf+COKp9zgb7T/CYOy5mpIUrZ7k0uAjixua5M+8DMUJ5W+KMNgqikLw38FYIdTVT86gEQVCoZsQBeoZr3dgeCQFWoE20cjA4XOW3ceIKDzBp0mixOMELpwgRsb3+CorwkqC","schnorr_public_key":"CN7uw1Ug54nhHVJMEm4m4o1nCXy9ajnDbGVne0fUjzODY6bMJC//NQETgkSSvNFa","seed":"stable weapon soap cruise future desk retire ho

## User Registration in Zenflows

In [8]:
# Read or get id of the person
for user in USERS:
    get_id_person(USERS_FILE, users_data, user, endpoint=ENDPOINT)

## Location Registration and Assignment

In [9]:
# Read or get the location id and set user locations
for user in USERS:
    get_location_id(LOCS_FILE, users_data[user], locs_data, user, endpoint=ENDPOINT)
    set_user_location(USERS_FILE, users_data, locs_data, user, endpoint=ENDPOINT)

## Unit of Measurement Registration

In [10]:
# Get the ids of all units
get_unit_id(UNITS_FILE, users_data['tchibo'], units_data, 'piece', 'u_piece', 'om2:one', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['tchibo'], units_data, 'mass', 'kg', 'om2:kilogram', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['tchibo'], units_data, 'time', 'h', 'om2:hour', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['tchibo'], units_data, 'energy', 'kWh', 'om2:kilowattHour', endpoint=ENDPOINT)

## Process Definitions for LoCI LAMP Production

In [11]:
# Create the processes for LoCI LAMP production

# Process for creating KROMA KRAFT cardboard components
process_name = 'Create_locilamp_cardboard_components'
user_data = users_data['locilamp_manufacturer']
note = f"Creation of KROMA KRAFT cardboard lamelles and base by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Process for creating the lampshade
process_name = 'Create_locilamp_lampshade'
user_data = users_data['locilamp_manufacturer']
note = f"Creation of transparent acid-free paper lampshade by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Process for assembling electrical components
process_name = 'Create_locilamp_electrical'
user_data = users_data['locilamp_manufacturer']
note = f"Assembly of textile cable, E27 socket, and switch by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Process for creating the LOCI LAMP design
process_name = 'Create_locilamp_design'
user_data = users_data['locilamp_designer']
note = f"Creation of LOCI LAMP V 2.0 design by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Process for final assembly of LOCI LAMP
process_name = 'Assemble_locilamp'
user_data = users_data['tchibo']
note = f"Final assembly and packaging of LOCI LAMP by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Save process data to file
with open(PROCESS_FILE, 'w') as f:
    json.dump(process_data, f, indent=2)
print(f"Process data saved to {PROCESS_FILE}")

Process data saved to use_cases/locilamp/proxy.dpp-staging.dyne.im%2Fzenflows%2Fapi/process_data.json


## Resource Specification Registration

In [12]:
# Register all resource specifications for LOCI LAMP components

# Raw material: KROMA KRAFT Displaykarton (FSC-certified)
name = 'kroma_kraft_cardboard'
note = 'KROMA KRAFT Displaykarton - PVC-free, FSC-certified cardboard'
classification = 'https://www.wikidata.org/wiki/Q389782'  # Cardboard
default_unit_id = units_data['mass']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Raw material: Transparent acid-free paper (FSC)
name = 'acidfree_paper'
note = 'Transparent acid-free paper - FSC-certified'
classification = 'https://www.wikidata.org/wiki/Q11472'  # Paper
default_unit_id = units_data['mass']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Electrical components: Textile cable
name = 'textile_cable'
note = 'Textile cable H03VVH2-F 2×0.75 mm² - Polyester, PVC, Copper'
classification = 'https://www.wikidata.org/wiki/Q199647'  # Electrical cable
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Electrical components: E27 socket
name = 'e27_socket'
note = 'E27 lamp socket - Copper, PET, galvanized steel'
classification = 'https://www.wikidata.org/wiki/Q5406598'  # Light socket
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Electrical components: EU plug
name = 'eu_plug'
note = 'EU plug (2-PIN) - Copper and PVC'
classification = 'https://www.wikidata.org/wiki/Q190642'  # Electrical plug
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Electrical components: Toggle switch
name = 'toggle_switch'
note = 'Toggle switch - Copper and PP'
classification = 'https://www.wikidata.org/wiki/Q163607'  # Switch
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Semi-finished: Cardboard lamelles and base
name = 'locilamp_cardboard_components'
note = 'LOCI LAMP lamelles and core base from KROMA KRAFT cardboard'
classification = 'https://github.com/locilamp/cardboard-components'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Semi-finished: Lampshade
name = 'locilamp_lampshade'
note = 'LOCI LAMP transparent acid-free paper lampshade'
classification = 'https://github.com/locilamp/lampshade'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Semi-finished: Electrical assembly
name = 'locilamp_electrical_assembly'
note = 'LOCI LAMP electrical assembly - cable, socket, plug, switch'
classification = 'https://github.com/locilamp/electrical'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Design: LOCI LAMP design
name = 'locilamp_design'
note = 'LOCI LAMP V 2.0 design specification'
classification = 'https://github.com/locilamp/design'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_designer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Final product: LOCI LAMP
name = 'locilamp'
note = 'LOCI LAMP V 2.0 - Complete table lamp self-assembly kit'
classification = 'https://www.wikidata.org/wiki/Q1146001'  # Table lamp
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['tchibo'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Work specifications
name = 'design_work'
note = 'Specification for design work'
classification = 'https://www.wikidata.org/wiki/Q82604'
default_unit_id = units_data['time']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_designer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'manufacturing_work'
note = 'Specification for manufacturing work'
classification = 'https://www.wikidata.org/wiki/Q187939'
default_unit_id = units_data['time']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

print(f"✓ Registered {len(res_spec_data)} resource specifications")

✓ Registered 13 resource specifications


## Initial Raw Materials Creation

In [13]:
# Create the initial raw materials

# KROMA KRAFT cardboard (0.3 kg based on ~0.345 kg lamp weight)
res_name = 'kroma_kraft_cardboard'
amount = 0.3
get_resource(res_data, res_spec_data, res_name, users_data['locilamp_manufacturer'], event_seq, amount, endpoint=ENDPOINT)

# Acid-free paper for lampshade
res_name = 'acidfree_paper'
amount = 0.02
get_resource(res_data, res_spec_data, res_name, users_data['locilamp_manufacturer'], event_seq, amount, endpoint=ENDPOINT)

# Textile cable
res_name = 'textile_cable'
amount = 1
get_resource(res_data, res_spec_data, res_name, users_data['locilamp_manufacturer'], event_seq, amount, endpoint=ENDPOINT)

# E27 socket
res_name = 'e27_socket'
amount = 1
get_resource(res_data, res_spec_data, res_name, users_data['locilamp_manufacturer'], event_seq, amount, endpoint=ENDPOINT)

# EU plug
res_name = 'eu_plug'
amount = 1
get_resource(res_data, res_spec_data, res_name, users_data['locilamp_manufacturer'], event_seq, amount, endpoint=ENDPOINT)

# Toggle switch
res_name = 'toggle_switch'
amount = 1
get_resource(res_data, res_spec_data, res_name, users_data['locilamp_manufacturer'], event_seq, amount, endpoint=ENDPOINT)

# Save initial resources to file
with open(RES_FILE, 'w') as f:
    json.dump(res_data, f, indent=2)
print(f"✓ Initial resources saved to {RES_FILE}")

✓ Initial resources saved to use_cases/locilamp/proxy.dpp-staging.dyne.im%2Fzenflows%2Fapi/initial_resources.json


## Prepare and Upload Product Images

Create placeholder image files and upload them to the DPP service.

In [14]:
# Create placeholder image files if they don't exist
ASSETS_DIR.mkdir(exist_ok=True, parents=True)

# Define image files for LOCI LAMP
image_files = {
    'locilamp_main.webp': 'Main product image - LOCI LAMP overview',
    'locilamp_main.webp': 'Detail image - Components and design',
    'locilamp_main.webp': 'Detail image - Materials and assembly',
}

# Create placeholder files (replace these with actual images)
for filename, description in image_files.items():
    filepath = ASSETS_DIR / filename
    if not filepath.exists():
        # Create a simple placeholder (in production, use actual image files)
        placeholder_content = f"[PLACEHOLDER] {description}\nProduct: LOCI LAMP V 2.0\nBrand: Tchibo GmbH\n"
        filepath.write_text(placeholder_content)
        print(f"✓ Created placeholder: {filename}")
    else:
        print(f"✓ Image exists: {filename}")

print(f"\n✓ Image files prepared in {ASSETS_DIR}")

✓ Image exists: locilamp_main.webp

✓ Image files prepared in /Users/alcibiade/dyne/if/Interfacer-notebook/assets


## Upload Images to DPP Service

Upload the product images to the interfacer-dpp storage using signed requests.

In [15]:
# Upload images to DPP service using the Tchibo user credentials
# This mirrors the UploadFileOnDPP function from the GUI

user_for_upload = users_data['tchibo']
eddsa_public_key = user_for_upload['eddsa_public_key']
eddsa_private_key = user_for_upload['keyring']['eddsa']

print("Uploading images to DPP service...")
print(f"  Using credentials for: {user_for_upload['name']}")
print(f"  DPP endpoint: {DPP_URL}")
print()

for filename in image_files.keys():
    filepath = str(ASSETS_DIR / filename)
    try:
        # Call the actual upload function from if_gc1dpp
        attachment_response = upload_file_on_dpp(
            filepath,
            eddsa_public_key,
            eddsa_private_key,
            DPP_URL
        )
        images_data[filename] = attachment_response
        print(f"✓ Uploaded {filename}")
        print(f"    ID: {attachment_response.get('id', 'N/A')}")
        print(f"    URL: {attachment_response.get('url', 'N/A')}")
    except Exception as e:
        print(f"✗ Failed to upload {filename}: {e}")
        images_data[filename] = {'error': str(e), 'filename': filename}

# Save image upload data
with open(IMAGES_FILE, 'w') as f:
    json.dump(images_data, f, indent=2)
print(f"\n✓ Image data saved to {IMAGES_FILE}")

Uploading images to DPP service...
  Using credentials for: Tchibo GmbH
  DPP endpoint: https://proxy.dpp-staging.dyne.im/interfacer-dpp

File checksum: d761a1e3de2bc2e586dc7efe11a5fd500ce4f6d72512b2e618bf55c82413deff
✓ Uploaded locilamp_main.webp
    ID: N/A
    URL: N/A

✓ Image data saved to use_cases/locilamp/proxy.dpp-staging.dyne.im%2Fzenflows%2Fapi/images_data.json


## Component Production

Produce the semi-finished components from raw materials.

In [16]:
# Step 1: Produce cardboard lamelles and base from KROMA KRAFT
cur_res = action = event_note = amount = cur_pros = None
action = 'consume'
event_note = 'consume KROMA KRAFT cardboard for lamelles and base'
amount = 0.3
cur_pros = process_data['Create_locilamp_cardboard_components']
cur_res = res_data['kroma_kraft_cardboard']

event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})
event_seq.append({'ts': ts, 'process_id': cur_pros['id'], 'name': cur_pros['name']})

# Produce cardboard components
action = 'produce'
event_note = 'produce cardboard lamelles and core base'
amount = 1
res_data['locilamp_cardboard_components'] = {
    "res_ref_id": f'locilamp_cardboard_components-{random.randint(0, 10000)}',
    "name": 'LOCI LAMP cardboard lamelles and base',
    "spec_id": res_spec_data['locilamp_cardboard_components']['id']
}
cur_res = res_data['locilamp_cardboard_components']

event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, new_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})

print(f"✓ Cardboard components produced (ID: {cur_res['id']})")

✓ Cardboard components produced (ID: 06E085QZW71X1VNW9CG9E5452W)


In [17]:
# Step 2: Produce lampshade from acid-free paper
cur_res = action = event_note = amount = cur_pros = None
action = 'consume'
event_note = 'consume acid-free paper for lampshade'
amount = 0.02
cur_pros = process_data['Create_locilamp_lampshade']
cur_res = res_data['acidfree_paper']

event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})
event_seq.append({'ts': ts, 'process_id': cur_pros['id'], 'name': cur_pros['name']})

# Produce lampshade
action = 'produce'
event_note = 'produce transparent acid-free paper lampshade'
amount = 1
res_data['locilamp_lampshade'] = {
    "res_ref_id": f'locilamp_lampshade-{random.randint(0, 10000)}',
    "name": 'LOCI LAMP transparent lampshade',
    "spec_id": res_spec_data['locilamp_lampshade']['id']
}
cur_res = res_data['locilamp_lampshade']

event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, new_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})

print(f"✓ Lampshade produced (ID: {cur_res['id']})")

✓ Lampshade produced (ID: 06E085R3MVNYNZT0M4RS8R4DEG)


In [18]:
# Step 3: Assemble electrical components (cable, socket, plug, switch)
cur_res = action = event_note = amount = cur_pros = None
cur_pros = process_data['Create_locilamp_electrical']

# Consume textile cable
action = 'consume'
event_note = 'consume textile cable for electrical assembly'
amount = 1
cur_res = res_data['textile_cable']
event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})

# Consume E27 socket
action = 'consume'
event_note = 'consume E27 socket for electrical assembly'
amount = 1
cur_res = res_data['e27_socket']
event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})

# Consume EU plug
action = 'consume'
event_note = 'consume EU plug for electrical assembly'
amount = 1
cur_res = res_data['eu_plug']
event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})

# Consume toggle switch
action = 'consume'
event_note = 'consume toggle switch for electrical assembly'
amount = 1
cur_res = res_data['toggle_switch']
event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})
event_seq.append({'ts': ts, 'process_id': cur_pros['id'], 'name': cur_pros['name']})

# Produce electrical assembly
action = 'produce'
event_note = 'produce electrical assembly with cable, socket, plug, and switch'
amount = 1
res_data['locilamp_electrical_assembly'] = {
    "res_ref_id": f'locilamp_electrical_assembly-{random.randint(0, 10000)}',
    "name": 'LOCI LAMP electrical assembly',
    "spec_id": res_spec_data['locilamp_electrical_assembly']['id']
}
cur_res = res_data['locilamp_electrical_assembly']

event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, new_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})

print(f"✓ Electrical assembly produced (ID: {cur_res['id']})")

✓ Electrical assembly produced (ID: 06E085RD29W1JVQ1JHV0HQ1T4G)


In [19]:
# Step 4: Create the LOCI LAMP design
cur_res = action = event_note = amount = cur_pros = None
action = 'work'
event_note = 'design work for LOCI LAMP V 2.0'
cur_pros = process_data['Create_locilamp_design']
effort_spec = {}
effort_spec['unit_id'] = res_spec_data['design_work']['defaultUnit']
effort_spec['spec_id'] = res_spec_data['design_work']['id']
effort_spec['amount'] = 40  # 40 hours of design work

event_id, ts = create_event(users_data['locilamp_designer'], action, event_note, amount=0, process=cur_pros,
                 res_spec_data=res_spec_data, effort_spec=effort_spec, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'amount': effort_spec['amount']})
event_seq.append({'ts': ts, 'process_id': cur_pros['id'], 'name': cur_pros['name']})

# Produce design
action = 'produce'
event_note = 'produce LOCI LAMP V 2.0 design'
amount = 1
res_data['locilamp_design'] = {
    "res_ref_id": f'locilamp_design-{random.randint(0, 10000)}',
    "name": 'LOCI LAMP V 2.0 design',
    "spec_id": res_spec_data['locilamp_design']['id']
}
cur_res = res_data['locilamp_design']

event_id, ts = create_event(users_data['locilamp_designer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, new_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})

print(f"✓ LOCI LAMP design created (ID: {cur_res['id']})")

✓ LOCI LAMP design created (ID: 06E085RGPWJEYFGGHBTRE6N2F0)


## Transfer Components to Tchibo

Transfer all produced components to Tchibo for final assembly.

In [20]:
# Transfer cardboard components to Tchibo
cur_res = res_data['locilamp_cardboard_components']
note = 'Transfer cardboard components from manufacturer to Tchibo'
action = 'transfer'
amount = 1
event_id, ts = make_transfer(users_data['locilamp_manufacturer'], action, note, users_data['tchibo'], amount, cur_res, locs_data, res_spec_data, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})
print(f"✓ Cardboard components transferred to Tchibo")

# Transfer lampshade to Tchibo
cur_res = res_data['locilamp_lampshade']
note = 'Transfer lampshade from manufacturer to Tchibo'
action = 'transfer'
amount = 1
event_id, ts = make_transfer(users_data['locilamp_manufacturer'], action, note, users_data['tchibo'], amount, cur_res, locs_data, res_spec_data, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})
print(f"✓ Lampshade transferred to Tchibo")

# Transfer electrical assembly to Tchibo
cur_res = res_data['locilamp_electrical_assembly']
note = 'Transfer electrical assembly from manufacturer to Tchibo'
action = 'transfer'
amount = 1
event_id, ts = make_transfer(users_data['locilamp_manufacturer'], action, note, users_data['tchibo'], amount, cur_res, locs_data, res_spec_data, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})
print(f"✓ Electrical assembly transferred to Tchibo")

✓ Cardboard components transferred to Tchibo
✓ Lampshade transferred to Tchibo
✓ Electrical assembly transferred to Tchibo


## Save All Data

In [21]:
# Save all resources to file
with open(RES_FILE, 'w') as f:
    json.dump(res_data, f, indent=2)
print(f"✓ All resources saved to {RES_FILE}")

# Store the parsed product data for the production notebook
PRODUCT_DATA_FILE = get_filename('product_data.json', ENDPOINT, USE_CASE)
with open(PRODUCT_DATA_FILE, 'w') as f:
    json.dump(loci_lamp_data, f, indent=2)
print(f"✓ Product data saved to {PRODUCT_DATA_FILE}")

✓ All resources saved to use_cases/locilamp/proxy.dpp-staging.dyne.im%2Fzenflows%2Fapi/initial_resources.json
✓ Product data saved to use_cases/locilamp/proxy.dpp-staging.dyne.im%2Fzenflows%2Fapi/product_data.json


## Setup Complete - Summary

In [22]:
print("\n" + "="*60)
print("LOCI LAMP SETUP COMPLETE")
print("="*60)
print(f"\n✓ {len(users_data)} users registered")
print(f"✓ {len(locs_data)} locations created")
print(f"✓ {len(units_data)} units registered")
print(f"✓ {len(res_spec_data)} resource specifications created")
print(f"✓ {len(process_data)} processes defined")
print(f"✓ {len(res_data)} resources created")
print(f"✓ {len(images_data)} images uploaded to DPP service")

print(f"\nComponents produced:")
print(f"  - Cardboard lamelles and base (transferred to Tchibo)")
print(f"  - Transparent lampshade (transferred to Tchibo)")
print(f"  - Electrical assembly (transferred to Tchibo)")
print(f"  - LOCI LAMP V 2.0 design (created by designer)")

print(f"\nProduct info from CSV:")
po = loci_lamp_data.get('Product Overview', {})
print(f"  - Brand: {po.get('Brand Name', 'N/A')}")
print(f"  - Product: {po.get('Product Name', 'N/A')}")
print(f"  - Model: {po.get('Model Name', 'N/A')}")
print(f"  - Country: {po.get('Country of Origin', 'N/A')}")

print(f"\n✓ You can now run the Production notebook to create the LOCI LAMP with DPP.")


LOCI LAMP SETUP COMPLETE

✓ 3 users registered
✓ 3 locations created
✓ 4 units registered
✓ 13 resource specifications created
✓ 5 processes defined
✓ 10 resources created
✓ 1 images uploaded to DPP service

Components produced:
  - Cardboard lamelles and base (transferred to Tchibo)
  - Transparent lampshade (transferred to Tchibo)
  - Electrical assembly (transferred to Tchibo)
  - LOCI LAMP V 2.0 design (created by designer)

Product info from CSV:
  - Brand: Tchibo GmbH
  - Product: LOCI LAMP
  - Model: LOCILAMP V 2.0
  - Country: Deutschland

✓ You can now run the Production notebook to create the LOCI LAMP with DPP.
